# Homework 5 — F1 Model Deployment

Predict whether an F1 driver scores points. Two models (Logistic Regression, Random Forest), tracked in MLflow, predictions written to `gr5069.yk3167`.

## Load data

F1 datasets from the course volume.

In [0]:
import pandas as pd
import numpy as np

path = "/Volumes/gr5069/raw/f1_data/"

results    = spark.read.csv(path + "results.csv",          header=True, inferSchema=True).toPandas()
races      = spark.read.csv(path + "races.csv",            header=True, inferSchema=True).toPandas()
drivers    = spark.read.csv(path + "driver_standings.csv", header=True, inferSchema=True).toPandas()
qualifying = spark.read.csv(path + "qualifying.csv",       header=True, inferSchema=True).toPandas()

print(f"results:    {results.shape}")
print(f"races:      {races.shape}")
print(f"drivers:    {drivers.shape}")
print(f"qualifying: {qualifying.shape}")
print("Data loaded.")

In [0]:
print("Columns:", results.columns.tolist())
print()
print(results.head(3))
print()
print("Position 分布（前 15）:")
print(results['position'].value_counts().head(15))
print()
print("Points 分布（前 15）:")
print(results['points'].value_counts().head(15))

In [0]:
spark.sql("SHOW SCHEMAS IN gr5069").show()

## Setup

Catalog and table names for storing predictions.

In [0]:
MY_SCHEMA = "yk3167"
CATALOG = "gr5069"
TABLE_LR = f"{CATALOG}.{MY_SCHEMA}.f1_predictions_lr"
TABLE_RF = f"{CATALOG}.{MY_SCHEMA}.f1_predictions_rf"

print("LR table:", TABLE_LR)
print("RF table:", TABLE_RF)

## Feature engineering

Merge results with race metadata. Target: `scored_points = points > 0`.

In [0]:
df = results.merge(
    races[['raceId', 'year', 'round', 'circuitId']],
    on='raceId',
    how='left'
)

df['scored_points'] = (df['points'] > 0).astype(int)

df = df[['resultId', 'raceId', 'driverId', 'constructorId',
         'grid', 'year', 'round', 'circuitId', 'scored_points']].copy()

df = df[df['grid'] > 0]
df = df.dropna()

print("Shape:", df.shape)
print()
print("Target 分布:")
print(df['scored_points'].value_counts(normalize=True).round(3))
print()
print(df.head())

## Train/test split

80/20, stratified on the target.

In [0]:
from sklearn.model_selection import train_test_split

feature_cols = ['grid', 'constructorId', 'driverId', 'year', 'round', 'circuitId']
X = df[feature_cols]
y = df['scored_points']

ids = df[['resultId', 'raceId', 'driverId', 'constructorId']].reset_index(drop=True)

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, ids,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train target 分布:", y_train.mean().round(3))
print("Test target 分布: ", y_test.mean().round(3))

## Create prediction tables

Two empty tables in `gr5069.yk3167`, one per model.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE_LR}")
spark.sql(f"DROP TABLE IF EXISTS {TABLE_RF}")

spark.sql(f"""
CREATE TABLE {TABLE_LR} (
    resultId       INT,
    raceId         INT,
    driverId       INT,
    constructorId  INT,
    actual         INT,
    predicted      INT,
    pred_proba     DOUBLE,
    model_name     STRING,
    run_id         STRING
)
""")

spark.sql(f"""
CREATE TABLE {TABLE_RF} (
    resultId       INT,
    raceId         INT,
    driverId       INT,
    constructorId  INT,
    actual         INT,
    predicted      INT,
    pred_proba     DOUBLE,
    model_name     STRING,
    run_id         STRING
)
""")

print("Tables created:")
spark.sql(f"SHOW TABLES IN {CATALOG}.{MY_SCHEMA}").filter("tableName LIKE 'f1_predictions%'").show()

## MLflow setup

One experiment for both runs.

In [0]:
import mlflow

EXPERIMENT_NAME = "/Users/yk3167@columbia.edu/homework5_f1_experiment"
mlflow.set_experiment(EXPERIMENT_NAME)
print("Experiment set:", EXPERIMENT_NAME)

## Model 1 — Logistic Regression

Standardized features, L2 penalty. Logs params, 4 metrics, model, confusion matrix, ROC curve.

In [0]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_params = {
    "C": 1.0,
    "penalty": "l2",
    "solver": "lbfgs",
    "max_iter": 1000,
    "random_state": 42
}

with mlflow.start_run(run_name="logistic_regression") as run:
    lr_run_id = run.info.run_id
    print("Run ID:", lr_run_id)

    mlflow.log_params(lr_params)
    mlflow.log_param("scaler", "StandardScaler")
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("n_train", X_train.shape[0])

    lr_model = LogisticRegression(**lr_params)
    lr_model.fit(X_train_scaled, y_train)

    y_pred_lr = lr_model.predict(X_test_scaled)
    y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

    metrics = {
        "accuracy":  accuracy_score(y_test, y_pred_lr),
        "precision": precision_score(y_test, y_pred_lr),
        "recall":    recall_score(y_test, y_pred_lr),
        "f1":        f1_score(y_test, y_pred_lr)
    }
    mlflow.log_metrics(metrics)
    print("Metrics:", {k: round(v, 4) for k, v in metrics.items()})

    mlflow.sklearn.log_model(lr_model, "model")

    cm = confusion_matrix(y_test, y_pred_lr)
    fig_cm, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Points", "Points"],
                yticklabels=["No Points", "Points"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Logistic Regression - Confusion Matrix")
    plt.tight_layout()
    cm_path = "/tmp/lr_confusion_matrix.png"
    fig_cm.savefig(cm_path)
    plt.close(fig_cm)
    mlflow.log_artifact(cm_path)

    fpr, tpr, _ = roc_curve(y_test, y_proba_lr)
    roc_auc = auc(fpr, tpr)
    fig_roc, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Logistic Regression - ROC Curve")
    ax.legend()
    plt.tight_layout()
    roc_path = "/tmp/lr_roc_curve.png"
    fig_roc.savefig(roc_path)
    plt.close(fig_roc)
    mlflow.log_artifact(roc_path)

    print("LR run logged. Run ID:", lr_run_id)

### Write LR predictions to Unity Catalog

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

predictions_lr = ids_test.copy()
predictions_lr["actual"]     = y_test.astype("int32").values
predictions_lr["predicted"]  = pd.Series(y_pred_lr).astype("int32").values
predictions_lr["pred_proba"] = pd.Series(y_proba_lr).astype("float64").values
predictions_lr["model_name"] = "logistic_regression"
predictions_lr["run_id"]     = lr_run_id

predictions_lr = predictions_lr.astype({
    "resultId":      "int32",
    "raceId":        "int32",
    "driverId":      "int32",
    "constructorId": "int32",
    "actual":        "int32",
    "predicted":     "int32",
    "pred_proba":    "float64",
    "model_name":    "string",
    "run_id":        "string",
})

schema = StructType([
    StructField("resultId",      IntegerType(), True),
    StructField("raceId",        IntegerType(), True),
    StructField("driverId",      IntegerType(), True),
    StructField("constructorId", IntegerType(), True),
    StructField("actual",        IntegerType(), True),
    StructField("predicted",     IntegerType(), True),
    StructField("pred_proba",    DoubleType(),  True),
    StructField("model_name",    StringType(),  True),
    StructField("run_id",        StringType(),  True),
])

predictions_lr_spark = spark.createDataFrame(predictions_lr, schema=schema)
predictions_lr_spark.write.mode("append").saveAsTable(TABLE_LR)

print(f"Inserted {predictions_lr.shape[0]} rows into {TABLE_LR}")
spark.sql(f"SELECT * FROM {TABLE_LR} LIMIT 5").show()
spark.sql(f"SELECT COUNT(*) AS n FROM {TABLE_LR}").show()

## Model 2 — Random Forest

200 trees, depth 12. Logs params, 4 metrics, model, confusion matrix, feature importance.

In [0]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    "n_estimators": 200,
    "max_depth": 12,
    "min_samples_split": 10,
    "min_samples_leaf": 4,
    "max_features": "sqrt",
    "random_state": 42,
    "n_jobs": -1
}

with mlflow.start_run(run_name="random_forest") as run:
    rf_run_id = run.info.run_id
    print("Run ID:", rf_run_id)

    mlflow.log_params(rf_params)
    mlflow.log_param("scaler", "None")
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("n_train", X_train.shape[0])

    rf_model = RandomForestClassifier(**rf_params)
    rf_model.fit(X_train, y_train)

    y_pred_rf = rf_model.predict(X_test)
    y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy":  accuracy_score(y_test, y_pred_rf),
        "precision": precision_score(y_test, y_pred_rf),
        "recall":    recall_score(y_test, y_pred_rf),
        "f1":        f1_score(y_test, y_pred_rf)
    }
    mlflow.log_metrics(metrics)
    print("Metrics:", {k: round(v, 4) for k, v in metrics.items()})

    mlflow.sklearn.log_model(rf_model, "model")

    cm = confusion_matrix(y_test, y_pred_rf)
    fig_cm, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", ax=ax,
                xticklabels=["No Points", "Points"],
                yticklabels=["No Points", "Points"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Random Forest - Confusion Matrix")
    plt.tight_layout()
    cm_path = "/tmp/rf_confusion_matrix.png"
    fig_cm.savefig(cm_path)
    plt.close(fig_cm)
    mlflow.log_artifact(cm_path)

    fi = pd.DataFrame({
        "feature": feature_cols,
        "importance": rf_model.feature_importances_
    }).sort_values("importance", ascending=True)

    fig_fi, ax = plt.subplots(figsize=(6, 4))
    ax.barh(fi["feature"], fi["importance"], color="seagreen")
    ax.set_xlabel("Importance")
    ax.set_title("Random Forest - Feature Importance")
    plt.tight_layout()
    fi_path = "/tmp/rf_feature_importance.png"
    fig_fi.savefig(fi_path)
    plt.close(fig_fi)
    mlflow.log_artifact(fi_path)

    print("RF run logged. Run ID:", rf_run_id)

### Write RF predictions to Unity Catalog

In [0]:
predictions_rf = ids_test.copy()
predictions_rf["actual"]     = y_test.astype("int32").values
predictions_rf["predicted"]  = pd.Series(y_pred_rf).astype("int32").values
predictions_rf["pred_proba"] = pd.Series(y_proba_rf).astype("float64").values
predictions_rf["model_name"] = "random_forest"
predictions_rf["run_id"]     = rf_run_id

predictions_rf = predictions_rf.astype({
    "resultId":      "int32",
    "raceId":        "int32",
    "driverId":      "int32",
    "constructorId": "int32",
    "actual":        "int32",
    "predicted":     "int32",
    "pred_proba":    "float64",
    "model_name":    "string",
    "run_id":        "string",
})

predictions_rf_spark = spark.createDataFrame(predictions_rf, schema=schema)
predictions_rf_spark.write.mode("append").saveAsTable(TABLE_RF)

print(f"Inserted {predictions_rf.shape[0]} rows into {TABLE_RF}")
spark.sql(f"SELECT * FROM {TABLE_RF} LIMIT 5").show()
spark.sql(f"SELECT COUNT(*) AS n FROM {TABLE_RF}").show()

## Verification

Sanity check: SQL accuracy from the stored predictions should match the MLflow metrics.

In [0]:
print("=" * 60)
print("HOMEWORK 5 — FINAL VERIFICATION")
print("=" * 60)

print(f"\nLR run_id: {lr_run_id}")
print(f"RF run_id: {rf_run_id}")

print(f"\n--- {TABLE_LR} ---")
spark.sql(f"""
SELECT model_name, COUNT(*) AS n_rows,
       AVG(actual)    AS actual_rate,
       AVG(predicted) AS predicted_rate,
       AVG(CASE WHEN actual = predicted THEN 1 ELSE 0 END) AS accuracy
FROM {TABLE_LR}
GROUP BY model_name
""").show()

print(f"--- {TABLE_RF} ---")
spark.sql(f"""
SELECT model_name, COUNT(*) AS n_rows,
       AVG(actual)    AS actual_rate,
       AVG(predicted) AS predicted_rate,
       AVG(CASE WHEN actual = predicted THEN 1 ELSE 0 END) AS accuracy
FROM {TABLE_RF}
GROUP BY model_name
""").show()

print("Done.")

In [0]:
print(f"=== {TABLE_LR} ===")
spark.sql(f"SELECT COUNT(*) AS n_rows FROM {TABLE_LR}").show()
spark.sql(f"SELECT * FROM {TABLE_LR} LIMIT 5").show(truncate=False)

print(f"=== {TABLE_RF} ===")
spark.sql(f"SELECT COUNT(*) AS n_rows FROM {TABLE_RF}").show()
spark.sql(f"SELECT * FROM {TABLE_RF} LIMIT 5").show(truncate=False)